In [ ]:
!pip install -q fastapi uvicorn nest-asyncio pandas numpy scikit-learn xgboost joblib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

MODEL_PATHS = {
    "xgboost": "/content/drive/MyDrive/DATN/xgboost_smote_mfcm_model.pkl",
    "random_forest": "/content/drive/MyDrive/DATN/rf_smote_mfcm_model.pkl"
}

model_artifacts = {}

for model_name, model_path in MODEL_PATHS.items():
    if os.path.exists(model_path):
        model_artifacts[model_name] = joblib.load(model_path)
        print(f"Đã load model: {model_name}")
    else:
        print(f"Chưa tìm thấy model: {model_name}")
        print("Đường dẫn:", model_path)

print("\nCác model hiện có:", list(model_artifacts.keys()))

Đã load model: xgboost
Đã load model: random_forest

Các model hiện có: ['xgboost', 'random_forest']


In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Dict, List, Any, Optional

# =============================
# 1. Khởi tạo API
# =============================

app = FastAPI(
    title="Fetal Health Prediction API",
    description="API dự đoán fetal_health bằng nhiều model: XGBoost, RandomForest + SMOTE + MFCM",
    version="2.0.0"
)


# =============================
# 2. Input schema
# =============================

class PredictRequest(BaseModel):
    model_name: str = "xgboost"
    features: Dict[str, float]


class BatchPredictRequest(BaseModel):
    model_name: str = "xgboost"
    data: List[Dict[str, float]]


# =============================
# 3. Hàm lấy model
# =============================

def get_artifact(model_name: str):
    model_name = model_name.lower().strip()

    if model_name not in model_artifacts:
        raise HTTPException(
            status_code=400,
            detail={
                "message": "Model không tồn tại hoặc chưa được load.",
                "model_name": model_name,
                "available_models": list(model_artifacts.keys())
            }
        )

    return model_artifacts[model_name]


# =============================
# 4. Hàm MFCM
# =============================

def make_weight_array(W, length):
    if length == 0:
        return np.array([])

    if isinstance(W, (int, float, np.integer, np.floating)):
        return np.full(length, float(W))

    W = np.asarray(W).reshape(-1)

    if W.size == 1:
        return np.full(length, float(W[0]))

    if W.size != length:
        return np.ones(length)

    return W.astype(float)


def compute_distances(
    DataCV,
    V,
    Binary_Col,
    Nominal_Col,
    Ordinal_Col,
    Interval_Col,
    W_b,
    W_n,
    W_o,
    W_i,
    p=2
):
    DataCV = np.asarray(DataCV, dtype=float)
    V = np.asarray(V, dtype=float)

    n_samples = DataCV.shape[0]
    n_clusters = V.shape[0]

    distances = np.zeros((n_samples, n_clusters))

    Binary_Col = list(Binary_Col)
    Nominal_Col = list(Nominal_Col)
    Ordinal_Col = list(Ordinal_Col)
    Interval_Col = list(Interval_Col)

    W_b_arr = make_weight_array(W_b, len(Binary_Col))
    W_n_arr = make_weight_array(W_n, len(Nominal_Col))
    W_o_arr = make_weight_array(W_o, len(Ordinal_Col))
    W_i_arr = make_weight_array(W_i, len(Interval_Col))

    for i in range(n_samples):
        for k in range(n_clusters):
            total = 0.0

            if len(Binary_Col) > 0:
                diff = DataCV[i, Binary_Col] != V[k, Binary_Col]
                total += np.sum(W_b_arr * diff.astype(float))

            if len(Nominal_Col) > 0:
                diff = DataCV[i, Nominal_Col] != V[k, Nominal_Col]
                total += np.sum(W_n_arr * diff.astype(float))

            if len(Ordinal_Col) > 0:
                diff = np.abs(DataCV[i, Ordinal_Col] - V[k, Ordinal_Col]) ** p
                total += np.sum(W_o_arr * diff)

            if len(Interval_Col) > 0:
                diff = np.abs(DataCV[i, Interval_Col] - V[k, Interval_Col]) ** p
                total += np.sum(W_i_arr * diff)

            distances[i, k] = total ** (1 / p)

    return distances


def predict_mfcm_membership(
    DataCV,
    V,
    Binary_Col,
    Nominal_Col,
    Ordinal_Col,
    Interval_Col,
    W_b,
    W_n,
    W_o,
    W_i,
    m=2,
    p=2
):
    distances = compute_distances(
        DataCV,
        V,
        Binary_Col,
        Nominal_Col,
        Ordinal_Col,
        Interval_Col,
        W_b,
        W_n,
        W_o,
        W_i,
        p
    )

    n = DataCV.shape[0]
    c = V.shape[0]

    U_pred = np.zeros((n, c))

    for i in range(n):
        if np.any(distances[i] == 0):
            zero_index = np.where(distances[i] == 0)[0]
            U_pred[i, zero_index] = 1
            continue

        for k in range(c):
            denom = np.sum((distances[i, k] / distances[i, :]) ** (2 / (m - 1)))
            U_pred[i, k] = 1 / denom

    return U_pred


# =============================
# 5. Chuẩn bị input theo từng model
# =============================

def prepare_input(df: pd.DataFrame, artifact: dict) -> pd.DataFrame:
    scaler = artifact["scaler"]

    V_final = artifact["V_final"]

    Binary_Col = artifact["Binary_Col"]
    Nominal_Col = artifact["Nominal_Col"]
    Ordinal_Col = artifact["Ordinal_Col"]
    Interval_Col = artifact["Interval_Col"]

    W_b = artifact["W_b"]
    W_n = artifact["W_n"]
    W_o = artifact["W_o"]
    W_i = artifact["W_i"]

    m = artifact["m"]
    p = artifact["p"]

    feature_columns_original = artifact["feature_columns_original"]
    feature_columns_final = artifact["feature_columns_final"]
    mfcm_cols = artifact["mfcm_cols"]

    missing_cols = [
        col for col in feature_columns_original
        if col not in df.columns
    ]

    if missing_cols:
        raise HTTPException(
            status_code=400,
            detail={
                "message": "Thiếu cột đầu vào",
                "missing_columns": missing_cols
            }
        )

    X_new = df[feature_columns_original].copy()

    try:
        X_new = X_new.astype(float)
    except Exception:
        raise HTTPException(
            status_code=400,
            detail="Dữ liệu đầu vào phải là số."
        )

    # Scale giống lúc train
    X_new_scaled = pd.DataFrame(
        scaler.transform(X_new),
        columns=feature_columns_original,
        index=X_new.index
    )

    # Tính membership MFCM
    U_new_mfcm = predict_mfcm_membership(
        DataCV=X_new_scaled.values,
        V=V_final,
        Binary_Col=Binary_Col,
        Nominal_Col=Nominal_Col,
        Ordinal_Col=Ordinal_Col,
        Interval_Col=Interval_Col,
        W_b=W_b,
        W_n=W_n,
        W_o=W_o,
        W_i=W_i,
        m=m,
        p=p
    )

    U_new_df = pd.DataFrame(
        U_new_mfcm,
        columns=mfcm_cols,
        index=X_new_scaled.index
    )

    # Ghép feature scale + membership MFCM
    X_new_final = pd.concat([X_new_scaled, U_new_df], axis=1)

    # Sắp xếp đúng thứ tự cột như lúc train
    X_new_final = X_new_final[feature_columns_final]

    return X_new_final


# =============================
# 6. Hàm predict theo model_name
# =============================

def predict_dataframe(df: pd.DataFrame, model_name: str):
    artifact = get_artifact(model_name)

    selected_model = artifact["model"]
    class_names = artifact.get(
        "class_names",
        {
            1: "Bình thường",
            2: "Nghi ngờ",
            3: "Bệnh lý"
        }
    )

    # Với XGBoost: label_offset = 1
    # Với RandomForest: label_offset = 0
    label_offset = artifact.get("label_offset", 0)

    X_new_final = prepare_input(df, artifact)

    y_pred_raw = selected_model.predict(X_new_final)

    y_pred_raw = np.asarray(y_pred_raw).astype(int)

    # Đổi nhãn nếu cần
    y_pred_final = y_pred_raw + label_offset

    results = []

    for i, pred in enumerate(y_pred_final):
        pred_int = int(pred)

        label = class_names.get(pred_int)
        if label is None:
            label = class_names.get(str(pred_int), "Không xác định")

        results.append({
            "index": int(i),
            "model_name": model_name,
            "prediction_raw": int(y_pred_raw[i]),
            "prediction": pred_int,
            "prediction_label": label
        })

    return results


# =============================
# 7. Endpoint API
# =============================

@app.get("/")
def home():
    return {
        "message": "API đang chạy",
        "available_models": list(model_artifacts.keys())
    }


@app.get("/models")
def get_models():
    return {
        "available_models": list(model_artifacts.keys()),
        "usage": {
            "xgboost": {
                "model_name": "xgboost",
                "note": "XGBoost trả nhãn 0,1,2 nên API cộng label_offset = 1"
            },
            "random_forest": {
                "model_name": "random_forest",
                "note": "RandomForest trả nhãn gốc 1,2,3 nên label_offset = 0"
            }
        }
    }


@app.get("/features")
def get_features(model_name: str = "xgboost"):
    artifact = get_artifact(model_name)

    return {
        "model_name": model_name,
        "required_features": artifact["feature_columns_original"],
        "total_features": len(artifact["feature_columns_original"])
    }


@app.post("/predict")
def predict_one(request: PredictRequest):
    df = pd.DataFrame([request.features])

    result = predict_dataframe(
        df=df,
        model_name=request.model_name
    )

    return {
        "status": "success",
        "model_name": request.model_name,
        "result": result[0]
    }


@app.post("/predict-batch")
def predict_batch(request: BatchPredictRequest):
    if len(request.data) == 0:
        raise HTTPException(
            status_code=400,
            detail="Danh sách data không được rỗng."
        )

    df = pd.DataFrame(request.data)

    results = predict_dataframe(
        df=df,
        model_name=request.model_name
    )

    return {
        "status": "success",
        "model_name": request.model_name,
        "total": len(results),
        "results": results
    }

In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

thread = threading.Thread(target=run_api)
thread.start()

Test api

In [ ]:
import requests

response = requests.get("http://127.0.0.1:8000/")
response.json()

INFO:     Started server process [13442]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:35350 - "GET / HTTP/1.1" 200 OK


{'message': 'API đang chạy', 'available_models': ['xgboost', 'random_forest']}

In [ ]:
response = requests.get("http://127.0.0.1:8000/features")
response.json()

INFO:     127.0.0.1:35356 - "GET /features HTTP/1.1" 200 OK


{'model_name': 'xgboost',
 'required_features': ['baseline value',
  'accelerations',
  'fetal_movement',
  'uterine_contractions',
  'light_decelerations',
  'severe_decelerations',
  'prolongued_decelerations',
  'abnormal_short_term_variability',
  'mean_value_of_short_term_variability',
  'percentage_of_time_with_abnormal_long_term_variability',
  'mean_value_of_long_term_variability',
  'histogram_width',
  'histogram_min',
  'histogram_max',
  'histogram_number_of_peaks',
  'histogram_number_of_zeroes',
  'histogram_mode',
  'histogram_mean',
  'histogram_median',
  'histogram_variance',
  'histogram_tendency'],
 'total_features': 21}

In [ ]:
import requests

url = "http://127.0.0.1:8000/predict"

data = {
    "model_name": "xgboost",
    "features": {
        "baseline value": 120,
        "accelerations": 0.003,
        "fetal_movement": 0,
        "uterine_contractions": 0.004,
        "light_decelerations": 0,
        "severe_decelerations": 0,
        "prolongued_decelerations": 0,
        "abnormal_short_term_variability": 73,
        "mean_value_of_short_term_variability": 0.5,
        "percentage_of_time_with_abnormal_long_term_variability": 43,
        "mean_value_of_long_term_variability": 2.4,
        "histogram_width": 64,
        "histogram_min": 62,
        "histogram_max": 126,
        "histogram_number_of_peaks": 2,
        "histogram_number_of_zeroes": 0,
        "histogram_mode": 120,
        "histogram_mean": 137,
        "histogram_median": 121,
        "histogram_variance": 73,
        "histogram_tendency": 1
    }
}

response = requests.post(url, json=data)
response.json()

INFO:     127.0.0.1:35360 - "POST /predict HTTP/1.1" 200 OK


{'status': 'success',
 'model_name': 'xgboost',
 'result': {'index': 0,
  'model_name': 'xgboost',
  'prediction_raw': 0,
  'prediction': 1,
  'prediction_label': 'Bình thường'}}

In [ ]:
import requests

url = "http://127.0.0.1:8000/predict"

data = {
    "model_name": "random_forest",
    "features": {
        "baseline value": 120,
        "accelerations": 0.003,
        "fetal_movement": 0,
        "uterine_contractions": 0.004,
        "light_decelerations": 0,
        "severe_decelerations": 0,
        "prolongued_decelerations": 0,
        "abnormal_short_term_variability": 73,
        "mean_value_of_short_term_variability": 0.5,
        "percentage_of_time_with_abnormal_long_term_variability": 43,
        "mean_value_of_long_term_variability": 2.4,
        "histogram_width": 64,
        "histogram_min": 62,
        "histogram_max": 126,
        "histogram_number_of_peaks": 2,
        "histogram_number_of_zeroes": 0,
        "histogram_mode": 120,
        "histogram_mean": 137,
        "histogram_median": 121,
        "histogram_variance": 73,
        "histogram_tendency": 1
    }
}

response = requests.post(url, json=data)
response.json()

INFO:     127.0.0.1:35376 - "POST /predict HTTP/1.1" 200 OK


{'status': 'success',
 'model_name': 'random_forest',
 'result': {'index': 0,
  'model_name': 'random_forest',
  'prediction_raw': 1,
  'prediction': 1,
  'prediction_label': 'Bình thường'}}